# NBA history: four visual explorations

Run all cells using this project's Pixi kernel. Queries and cleaning live in
`nba/wrangling.py`; plotting functions live in `nba/visualisation.py`.
The database is opened read-only. Game charts use regular-season games.
Streaks use NBA seasons; attendance and scoring use calendar years.
Missing records limit historical coverage. Attendance cleaning is applied
automatically on every run; see the README for the excluded observations.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import HTML, display

from nba.wrangling import (
    connect_database, get_attendance, get_new_player_heights,
    get_scoring, get_streaks,
)
from nba.visualisation import (
    animate_new_player_heights, plot_attendance, plot_scoring, streak_season_selector,
)

# Find the project when launched from its root or a subfolder.
ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "data" / "nba.sqlite").is_file()), None
)
if ROOT is None:
    raise FileNotFoundError("Open the project folder containing data/nba.sqlite")
con = connect_database(ROOT / "data" / "nba.sqlite")


## 1. Winning and losing streaks

Note: this one takes about 70s to run.

Select an NBA season from the dropdown. Each team's line uses its game
number within that season. Winning streaks are positive, losing streaks
negative, and completed streaks reset vertically to zero. Streaks restart
each season; playoffs are excluded. The final recorded run stays open.

Game numbers reflect recorded games. Incomplete historical coverage can
undercount games and streaks, so these are not verified official records.


In [ ]:
streaks = get_streaks(con)
display(HTML(streak_season_selector(streaks)))


## 2. Highest and lowest attendance each year

Annual extremes use non-missing, non-negative attendance. The two known
erroneous 1995 observations are automatically excluded before aggregation;
the README records their IDs, values, and sources. The raw database is
preserved. Recorded zeros are retained, and other unusually large values
still require verification before being treated as records.


In [ ]:
attendance = get_attendance(con)
plot_attendance(attendance)
plt.show()


## 3. Average points per game and by quarter

Values combine both teams' points and are averaged by calendar year.
Full-game totals include overtime; quarter totals do not. Separate panels
keep both scales readable. Quarter averages use complete numeric quarter
records, so their sample can differ from the full-game sample. Missing
years are shown as gaps rather than zeros.


In [ ]:
scoring = get_scoring(con)
plot_scoring(scoring)
plt.show()


## 4. Heights of new players each year

Each frame groups newcomers by their first NBA year, rather than draft
year. Heights are converted to centimetres; fixed bins and axes support
comparisons. Percentages sum to 100% within each cohort. Use the player
controls to play, pause, or inspect a year.

Only players with available biographies and valid recorded heights are
included. Coverage is incomplete, and these heights were not necessarily
measured at debut. Small cohorts can produce irregular distributions.


In [ ]:
new_player_heights = get_new_player_heights(con)
height_fig, height_animation = animate_new_player_heights(new_player_heights)
display(HTML(height_animation.to_jshtml(fps=3, default_mode="loop")))
plt.close(height_fig)


In [ ]:
# Close the read-only connection when finished; rerun setup to reconnect.
con.close()
